<a href="https://colab.research.google.com/github/jagtapnakshatra/us-etf-fund-analytics/blob/main/etf_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
# =========================
# 1. SETUP + DRIVE MOUNT
# =========================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/Linkedin Projects/USFunds"
RAW = f"{BASE}/raw"
GOLD = f"{BASE}/gold"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [92]:
# =========================
# 2. LOAD DATA
# =========================

# ETF price data
etf_prices = pd.read_csv(f"{RAW}/ETF prices.csv")

# ETF metadata
etf_meta = pd.read_csv(f"{RAW}/ETFs.csv")

# Mutual fund price files
mf_files = [
    "MutualFund prices - A-E.csv",
    "MutualFund prices - F-K.csv",
    "MutualFund prices - L-P.csv",
    "MutualFund prices - Q-Z.csv"
]

mf_prices = pd.concat(
    [pd.read_csv(f"{RAW}/{f}") for f in mf_files],
    ignore_index=True
)

In [93]:
# =========================
# 3. CLEAN + SORT DATA
# =========================

# Convert date column to datetime
etf_prices["price_date"] = pd.to_datetime(etf_prices["price_date"])

# Sort for time-series calculations
etf_prices = etf_prices.sort_values(["fund_symbol", "price_date"])

In [94]:
# =========================
# 4. FEATURE ENGINEERING (RETURNS + VOLATILITY)
# =========================

# Previous close price per fund
etf_prices["prev_close"] = etf_prices.groupby("fund_symbol")["close"].shift(1)

# Daily returns
etf_prices["daily_return"] = (
    (etf_prices["close"] - etf_prices["prev_close"])
    / etf_prices["prev_close"]
)

# Remove infinite values
etf_prices["daily_return"] = etf_prices["daily_return"].replace([np.inf, -np.inf], np.nan)

# 30-day rolling volatility (optional feature)
etf_prices["volatility_30d"] = (
    etf_prices.groupby("fund_symbol")["daily_return"]
    .rolling(30)
    .std()
    .reset_index(level=0, drop=True)
)

In [95]:
# =========================
# 5. BUILD FUND-LEVEL GOLD TABLE
# =========================

gold_perf = etf_prices.groupby("fund_symbol").agg(
    avg_return=("daily_return", "mean"),
    volatility=("daily_return", "std"),
    avg_volume=("volume", "mean")
).reset_index()

# Merge ETF metadata
gold_perf = gold_perf.merge(etf_meta, on="fund_symbol", how="left")

In [96]:
# =========================
# 6. CLEAN MISSING VALUES
# =========================

# Replace missing category values
gold_perf["fund_category"] = gold_perf["fund_category"].fillna("Unknown")

In [97]:
# =========================
# 7. SCORE ENGINEERING (RANKING SYSTEM)
# =========================

# Percentile rank for returns (higher = better)
gold_perf["return_pctile"] = gold_perf["avg_return"].rank(pct=True)

# Percentile rank for risk (lower volatility is better → invert ranking)
gold_perf["risk_pctile"] = gold_perf["volatility"].rank(pct=True, ascending=False)

# Combined score (custom weighting)
gold_perf["overall_score"] = (
    0.6 * gold_perf["return_pctile"] +
    0.4 * gold_perf["risk_pctile"]
)

In [98]:
# =========================
# 8. RANKING + SEGMENTS
# =========================

# Rank funds
gold_perf["overall_rank"] = gold_perf["overall_score"].rank(
    ascending=False,
    method="dense"
)

# Segment funds into quartiles
gold_perf["fund_segment"] = pd.qcut(
    gold_perf["overall_score"],
    q=4,
    labels=["Bottom Quartile", "Below Average", "Above Average", "Top Quartile"]
)

In [99]:
# =========================
# 9. CREATE "HIDDEN GEM" LOGIC
# =========================

return_cutoff = gold_perf["avg_return"].median()
risk_cutoff = gold_perf["volatility"].median()
volume_cutoff = gold_perf["avg_volume"].median()

gold_perf["hidden_gem"] = (
    (gold_perf["avg_return"] > return_cutoff) &
    (gold_perf["volatility"] < risk_cutoff) &
    (gold_perf["avg_volume"] < volume_cutoff)
)

In [100]:
# =========================
# 10. OPPORTUNITY LABEL
# =========================

gold_perf["opportunity_type"] = "Normal"
gold_perf.loc[gold_perf["hidden_gem"], "opportunity_type"] = "Hidden Gem"

In [101]:
# =========================
# 11. CATEGORY SUMMARY TABLE
# =========================

category_summary = gold_perf.groupby("fund_category").agg(
    fund_count=("fund_symbol", "count"),
    avg_return=("avg_return", "mean"),
    median_return=("avg_return", "median"),
    avg_volatility=("volatility", "mean"),
    avg_score=("overall_score", "mean"),
    hidden_gems=("hidden_gem", "sum")
).reset_index()

# Add ranking of categories
category_summary["category_rank"] = category_summary["avg_score"].rank(
    ascending=False,
    method="dense"
)

In [102]:
gold_perf

,fund_symbol,avg_return,volatility,avg_volume,quote_type,region,fund_short_name,fund_long_name,currency,fund_category,...,fund_stdev_10years,fund_sharpe_ratio_10years,fund_treynor_ratio_10years,return_pctile,risk_pctile,overall_score,overall_rank,fund_segment,hidden_gem,opportunity_type
0,AAA,-0.000004,0.000513,2788.064516,ETF,US,BMO Pyrford Intl Stock Fd - Cla,BMO Pyrford International Stock Fund Class A,USD,Unknown,...,NaN,NaN,NaN,0.163636,0.996537,0.496797,1158.0,Below Average,False,Normal
1,AAAU,0.000534,0.009491,272544.819277,ETF,US,DWS RREEF Real Assets Fund - Cl,DWS RREEF Real Assets Fund - Class A,USD,Unknown,...,NaN,NaN,NaN,0.733766,0.679221,0.711948,202.0,Top Quartile,False,Normal
2,AADR,0.000401,0.012767,9591.264850,ETF,US,AllianzGI Health Sciences Fund,Virtus AllianzGI Health Sciences Fund Class P,USD,Foreign Large Growth,...,16.78,0.53,8.15,0.610823,0.461905,0.551255,823.0,Above Average,False,Normal
3,AAXJ,0.000285,0.016367,853555.960562,ETF,US,NaN,American Century One Choice Blend+ 2015 Portfo...,USD,Pacific/Asia ex-Japan Stk,...,16.83,0.36,4.81,0.499567,0.224675,0.389610,1615.0,Bottom Quartile,False,Normal
4,ABEQ,0.000238,0.012079,9677.872340,ETF,US,Thrivent Large Cap Growth Fund,Thrivent Large Cap Growth Fund Class A,USD,Large Value,...,NaN,NaN,NaN,0.451948,0.507359,0.474113,1290.0,Below Average,False,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2305,ZHDG,0.000274,0.005242,61658.252427,ETF,US,Boston Partners Emerging Market,Boston Partners Emerging Markets Fund Institut...,USD,Large Blend,...,NaN,NaN,NaN,0.485281,0.824675,0.621039,514.0,Top Quartile,False,Normal
2306,ZIG,0.000382,0.014872,7523.483670,ETF,US,Baillie Gifford Positive Change,Baillie Gifford Positive Change Equities Fund ...,USD,Unknown,...,NaN,NaN,NaN,0.591775,0.305628,0.477316,1277.0,Below Average,False,Normal
2307,ZIVZF,0.000559,0.021580,59763.285199,ETF,US,BlackRock U.S. Impact Fund Inve,BlackRock U.S. Impact Fund Investor A,USD,Trading--Miscellaneous,...,34.04,0.38,-3.85,0.759740,0.109524,0.499654,1136.0,Below Average,False,Normal
2308,ZROZ,0.000353,0.014391,36402.467917,ETF,US,Boston Partners Global Equity F,Boston Partners Global Equity Institutional Class,USD,Long Government,...,21.11,0.55,1.79,0.557576,0.339394,0.470303,1317.0,Below Average,False,Normal


In [103]:
# =========================
# 12. EXPORT FINAL TABLES
# =========================

gold_perf.to_csv(f"{GOLD}/gold_etf_table.csv", index=False)
category_summary.to_csv(f"{GOLD}/category_summary.csv", index=False)